# 🌍 Indic Stance Detection on Global Warming (Hindi & Bengali)
## Google MuRIL Transformer Fine-Tuning Pipeline
**Course:** CSET 346: Natural Language Processing | **Institution:** Bennett University  
**Instructor:** Dr. Supriya Chanda | **Author:** Tanmay Goyal (student@bennett.edu.in)  
**Target Claim:** *"Climate change and global warming is a serious concern."*  
**Model:** `google/muril-base-cased` (Multilingual Representations for Indic Languages)  
**Labels:** `FAVOUR` | `AGAINST` | `NONE`


### 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))


### 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn evaluate seaborn matplotlib


### 3. Load Project Datasets

In [ ]:
import os
import pandas as pd
import numpy as np
import re

# Check if files exist; if not, open a 1-click upload button
if not os.path.exists('train.csv'):
    print('Files not detected. Please upload the 4 CSV files from d:/nlp/data/processed/:')
    print('(train.csv, val.csv, test.csv, hidden_test_unlabeled.csv)')
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as e:
        print('Upload helper note:', e)

train_df = pd.read_csv('train.csv', keep_default_na=False)
val_df = pd.read_csv('val.csv', keep_default_na=False)
test_df = pd.read_csv('test.csv', keep_default_na=False)
hidden_df = pd.read_csv('hidden_test_unlabeled.csv', keep_default_na=False)

print(f'Successfully Loaded: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}, Hidden={len(hidden_df)}')
display(train_df.head(3))


### 4. Setup MuRIL Tokenizer with Emoji Vocabulary Expansion

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'google/muril-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Discover unique emojis to expand vocabulary (preventing collapse into [UNK])
emoji_pattern = re.compile(r'[\U00010000-\U0010ffff\u2600-\u27bf]')
all_texts = train_df['cleaned_text'].tolist() + test_df['cleaned_text'].tolist()
unique_emojis = sorted(list(set(emoji_pattern.findall(' '.join(all_texts)))))

num_added = tokenizer.add_tokens(unique_emojis)
print(f'Added {num_added} unique emoji tokens to MuRIL tokenizer! (Vocab: {len(tokenizer):,d})')


### 5. HuggingFace Dataset & Label Encoding

In [ ]:
from datasets import Dataset

TARGET_CLASSES = ['AGAINST', 'FAVOUR', 'NONE']
LABEL2ID = {label: idx for idx, label in enumerate(TARGET_CLASSES)}
ID2LABEL = {idx: label for idx, label in enumerate(TARGET_CLASSES)}

train_df['label'] = train_df['stance'].map(LABEL2ID)
val_df['label'] = val_df['stance'].map(LABEL2ID)
test_df['label'] = test_df['stance'].map(LABEL2ID)

def tokenize_batch(batch):
    return tokenizer(batch['cleaned_text'], padding='max_length', truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[['cleaned_text', 'label']]).map(tokenize_batch, batched=True)
val_ds = Dataset.from_pandas(val_df[['cleaned_text', 'label']]).map(tokenize_batch, batched=True)
test_ds = Dataset.from_pandas(test_df[['cleaned_text', 'label']]).map(tokenize_batch, batched=True)
hidden_ds = Dataset.from_pandas(hidden_df[['cleaned_text']]).map(tokenize_batch, batched=True)


### 6. Model Initialization & Token Embedding Resizing

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

if num_added > 0:
    model.resize_token_embeddings(len(tokenizer))
    print('Successfully resized model embeddings for newly added emojis!')


### 7. Training Arguments & Trainer Configuration

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='macro', zero_division=0)
    rec = recall_score(labels, preds, average='macro', zero_division=0)
    return {
        'macro_f1': round(macro_f1, 4),
        'accuracy': round(acc, 4),
        'precision': round(prec, 4),
        'recall': round(rec, 4)
    }

training_args = TrainingArguments(
    output_dir='./muril_results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


### 8. Fine-Tuning Execution

In [ ]:
train_result = trainer.train()
print('Fine-Tuning Complete!')
trainer.save_model('./best_muril_model')
tokenizer.save_pretrained('./best_muril_model')


### 9. Test Set Evaluation & Confusion Matrix Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

test_preds_raw = trainer.predict(test_ds)
test_preds = np.argmax(test_preds_raw.predictions, axis=-1)
test_labels = test_df['label'].values

print('=' * 60)
print('MuRIL TEST EVALUATION REPORT')
print('=' * 60)
print(classification_report(test_labels, test_preds, target_names=TARGET_CLASSES, digits=4))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
plt.title('Confusion Matrix: Google MuRIL Fine-Tuned', fontsize=12)
plt.xlabel('Predicted Stance')
plt.ylabel('True Stance')
plt.show()


### 10. Generate Competition Predictions (Hidden Test Set)

In [ ]:
hidden_preds_raw = trainer.predict(hidden_ds)
hidden_preds = np.argmax(hidden_preds_raw.predictions, axis=-1)
hidden_pred_labels = [ID2LABEL[p] for p in hidden_preds]

sub_df = pd.DataFrame({
    'id': hidden_df['id'],
    'predicted_stance': hidden_pred_labels
})
sub_df.to_csv('transformer_predictions.csv', index=False, encoding='utf-8-sig')
print(f'Saved {len(sub_df)} predictions to transformer_predictions.csv')
display(sub_df['predicted_stance'].value_counts())


### 11. Sarcasm & Semantic Minimization Stress-Test (Resolving TF-IDF Blindspots)

In [ ]:
stress_cases = [
    ('हाँ भाई, ग्लोबल वार्मिंग तो शायद मौसम का मूड स्विंग है 😂', 'FAVOUR (Sarcastic)'),
    ('पेड़ काटने से कुछ नहीं होता, प्रकृति खुद सब संभाल लेती है', 'AGAINST (Minimizing)'),
    ('গাছ কেটে ফেললে আমাদের কোনো ক্ষতি হবে না, সব ঠিক থাকবে', 'AGAINST (Bengali Minimizing)'),
    ('आज दिल्ली में बारिश हो रही है और हवा बहुत ठंडी है', 'NONE (Weather)'),
    ('कार्बन डाइऑक्साइड कम करना हमारी पहली जिम्मेदारी होनी चाहिए 🌱', 'FAVOUR (Clear)')
]

print('=' * 75)
print('MuRIL CONTEXTUAL SELF-ATTENTION EVALUATION ON TRICKY SAMPLES')
print('=' * 75)

pipe_inputs = tokenizer([text for text, _ in stress_cases], padding=True, truncation=True, return_tensors='pt').to(model.device)
with torch.no_grad():
    logits = model(**pipe_inputs).logits
    preds = torch.argmax(logits, dim=-1).cpu().numpy()

for (text, expected), pred_idx in zip(stress_cases, preds):
    pred_label = ID2LABEL[pred_idx]
    match = 'MATCH' if expected.startswith(pred_label) else 'MISMATCH'
    print(f'Text:      {text}')
    print(f'Expected:  {expected}')
    print(f'Predicted: {pred_label}  [{match}]')
    print('-' * 75)
